In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import requests

## Data Exploration

Before building our first pipeline and retrieving the selected dataset, some preliminary work is required.

### Objectives
- Request data from the API  
- Explore the data structure  
- Identify necessary transformations  

### Notes
Reviewing the API documentation is a key part of this phase, as it often answers questions that arise during exploration.

In [2]:
url = "https://opensky-network.org/api/states/all"
response = requests.get(url)
data = response.json()

In [3]:
aviation_raw = pd.DataFrame(data)
aviation_raw.head()

,time,states
0,1778068181,"[39de4e, TVF49MD , France, 1778068181, 1778068..."
1,1778068181,"[e8027c, LPE2132 , Chile, 1778068180, 17780681..."
2,1778068181,"[39de4c, TVF21CH , France, 1778068180, 1778068..."
3,1778068181,"[801645, AIC2YB , India, 1778068156, 17780681..."
4,1778068181,"[ab6fdd, AAL1464 , United States, 1778068181, ..."


In [4]:
aviation_raw.shape

(8206, 2)

## Data Transformation & Structure Overview

After retrieving the data from the API in JSON format, it is converted into a Pandas DataFrame.

The first five rows and the shape of the dataset are displayed to get a quick overview of the structure.

According to the OpenSky API documentation, the data consists of:

- **Timestamp (Unix time):** Represents the time of the data snapshot; all records share this value.
- **State vector:** Contains real-time information about each aircraft.

In [5]:
aviation_raw.iloc[0]

time                                             1778068181
states    [39de4e, TVF49MD , France, 1778068181, 1778068...
Name: 0, dtype: object

In [6]:
aviation_raw["time"].unique()

array([1778068181], dtype=int64)

In [7]:
aviation_raw["states"].iloc[0]

['39de4e',
 'TVF49MD ',
 'France',
 1778068181,
 1778068181,
 13.6822,
 40.3868,
 11285.22,
 False,
 243.27,
 132.26,
 0,
 None,
 11536.68,
 '4303',
 False,
 0]

In [8]:
count_el_state_vektor = len(aviation_raw["states"].iloc[0])
print(f"Number of elements per state vector: {len(aviation_raw["states"].iloc[0])}")


Number of elements per state vector: 17


After verifying that all timestamps in the dataset are identical, the state vector was explored by selecting the first row, displaying its content, and counting the number of elements it contains.

The result does not fully match the schema described in the OpenSky API documentation. Further review shows that unauthenticated or restricted requests return only 17 fields instead of the full 18.

The 18th field is optional and is only included under specific conditions. For the current analysis, this additional feature is not required, so the dataset is used as provided.

In [9]:
# defining categories as metadata is not transmitted on requests
categories = ['icao240', 'callsign', 'origin_country', 'time_position', 'last_contact', 'longitude', 'latitude',
              'baro_altitude', 'on_ground', 'velocity', 'true_track', 'vertical_rate', 'sensors', 'geo_altitude',
              'squawk', 'spi', 'position_source'
             ]

In [10]:
# building dataframe from state vectors and setting columns 
states_data = pd.DataFrame(data["states"], columns=categories)

In [11]:
states_data

,icao240,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,sensors,geo_altitude,squawk,spi,position_source
0,39de4e,TVF49MD,France,1.778068e+09,1778068181,13.6822,40.3868,11285.22,False,243.27,132.26,0.00,None,11536.68,4303,False,0
1,e8027c,LPE2132,Chile,1.778068e+09,1778068180,-78.4136,-9.3690,9555.48,False,221.58,154.27,8.13,None,10203.18,None,False,0
2,39de4c,TVF21CH,France,1.778068e+09,1778068180,15.9382,45.0861,11887.20,False,229.96,127.00,0.33,None,12047.22,0657,False,0
3,801645,AIC2YB,India,1.778068e+09,1778068156,83.7075,25.0177,11887.20,False,263.59,119.72,-0.33,None,12458.70,None,False,0
4,ab6fdd,AAL1464,United States,1.778068e+09,1778068181,-83.9312,41.3199,10363.20,False,169.63,273.83,1.95,None,10408.92,5700,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8201,c00741,WJA669,Canada,1.778068e+09,1778068180,-68.3347,46.8486,10972.80,False,185.59,294.22,-0.33,None,10972.80,3650,False,0
8202,aa781f,CKS216,United States,1.778068e+09,1778068180,-97.0611,48.4213,9753.60,False,240.83,310.50,0.00,None,9433.56,2253,False,0
8203,4b1903,SWR8T,Switzerland,1.778068e+09,1778068181,6.3739,47.7073,8183.88,False,231.37,285.21,5.20,None,8161.02,3041,False,0
8204,a5eba7,N4805G,United States,1.778068e+09,1778068178,-76.7185,37.1296,853.44,False,33.50,273.52,-0.33,None,853.44,None,False,0


After expanding the state vectors into 17 columns, we add the corresponding timestamps to complete the structured DataFrame.

In [12]:
aviation_expanded = states_data.copy()
aviation_expanded.insert(0, "time", aviation_raw.time)

In [13]:
aviation_expanded

,time,icao240,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,sensors,geo_altitude,squawk,spi,position_source
0,1778068181,39de4e,TVF49MD,France,1.778068e+09,1778068181,13.6822,40.3868,11285.22,False,243.27,132.26,0.00,None,11536.68,4303,False,0
1,1778068181,e8027c,LPE2132,Chile,1.778068e+09,1778068180,-78.4136,-9.3690,9555.48,False,221.58,154.27,8.13,None,10203.18,None,False,0
2,1778068181,39de4c,TVF21CH,France,1.778068e+09,1778068180,15.9382,45.0861,11887.20,False,229.96,127.00,0.33,None,12047.22,0657,False,0
3,1778068181,801645,AIC2YB,India,1.778068e+09,1778068156,83.7075,25.0177,11887.20,False,263.59,119.72,-0.33,None,12458.70,None,False,0
4,1778068181,ab6fdd,AAL1464,United States,1.778068e+09,1778068181,-83.9312,41.3199,10363.20,False,169.63,273.83,1.95,None,10408.92,5700,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8201,1778068181,c00741,WJA669,Canada,1.778068e+09,1778068180,-68.3347,46.8486,10972.80,False,185.59,294.22,-0.33,None,10972.80,3650,False,0
8202,1778068181,aa781f,CKS216,United States,1.778068e+09,1778068180,-97.0611,48.4213,9753.60,False,240.83,310.50,0.00,None,9433.56,2253,False,0
8203,1778068181,4b1903,SWR8T,Switzerland,1.778068e+09,1778068181,6.3739,47.7073,8183.88,False,231.37,285.21,5.20,None,8161.02,3041,False,0
8204,1778068181,a5eba7,N4805G,United States,1.778068e+09,1778068178,-76.7185,37.1296,853.44,False,33.50,273.52,-0.33,None,853.44,None,False,0


To get some understanding of what data values we are expecting, let's look at what pandas uses to store those.<br>We will be using this as a very braod frame to setup up initial validation of the fetched data, we will have to ge back to this at some point and tighten the schemas used but for now we are still exploring.

In [14]:
aviation_expanded.dtypes

time                 int64
icao240             object
callsign            object
origin_country      object
time_position      float64
last_contact         int64
longitude          float64
latitude           float64
baro_altitude      float64
on_ground             bool
velocity           float64
true_track         float64
vertical_rate      float64
sensors             object
geo_altitude       float64
squawk              object
spi                   bool
position_source      int64
dtype: object

longitude
-80.9708     3
 12.4705     2
-2.7087      2
-3.3567      2
-113.9886    2
            ..
-161.1341    1
-85.9235     1
-80.5980     1
-84.4368     1
-79.9140     1
Name: count, Length: 8044, dtype: int64